# Step 1: Install Infrastructure Engines
Installing system dependencies and Python packages for PDF processing, image conversion, and Word document generation.

In [40]:
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr
!pip install pypdf pdf2image python-docx pytesseract

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (24.02.0-1ubuntu9.11).
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove

# Step 2: Environment Setup and Initialization
Setting up input/output directories, loading the system prompt, and verifying the OpenAI client configuration.

In [41]:
import os
import io
import re
import glob
import base64
import shutil
from google.colab import userdata
from google.colab import drive
import pypdf
from pdf2image import convert_from_path
from docx import Document
from docx.shared import Inches
from openai import OpenAI

# 1. Clear conflicting local directories and mount Google Drive safely
print("Mounting Google Drive...")
target_mount = '/content/drive'
try:
    # Force unmount to clear any broken connections
    os.system('umount -f /content/drive')
except Exception:
    pass

if os.path.exists(target_mount):
    try:
        shutil.rmtree(target_mount)
        print("Successfully deleted conflicting /content/drive folder.")
    except Exception as e:
        os.system('rm -rf /content/drive')

try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"Warning during mounting: {e}")

# 2. Define paths
INPUT_DIR = "/content/drive/My Drive/New_Data/Input_pdf_with_drawings"
OUTPUT_DIR = "/content/drive/My Drive/New_Data/Data_Prep_to_docx"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Retrieve secure OpenAI key
try:
    openai_key = userdata.get('OPENAI_KEY')
except Exception:
    openai_key = os.environ.get('OPENAI_KEY', '')

# 4. Read system prompt for multimodal drawing analysis and clean Python wrappers if present
with open('/content/Enhanced prompt.txt', 'r', encoding='utf-8') as f:
    raw_prompt = f.read()

# Remove comment lines, assignment wrappers, and string literal decorations
cleaned = raw_prompt.strip()
# Remove leading Python comments
cleaned = re.sub(r'^\s*#.*$', '', cleaned, flags=re.MULTILINE)
# Remove 'prompt = textwrap.dedent("""' and any similar variable wrappers
cleaned = re.sub(r'^\s*(?:[a-zA-Z_]\w*\s*=\s*)?(?:textwrap\.dedent\(\s*)?["\']{3}\\?', '', cleaned.strip(), flags=re.IGNORECASE)
# Remove trailing triple quotes and closing parenthesis
cleaned = re.sub(r'["\']{3}\s*\)?\s*$', '', cleaned).strip()

SYSTEM_PROMPT = cleaned

# Initialize OpenAI Client
if openai_key:
    client = OpenAI(api_key=openai_key)
    print("Environment setup completed successfully. OpenAI Client initialized and ready.")
else:
    client = None
    print("Warning: OPENAI_KEY is missing. Code will fall back to simulation mode.")

Mounting Google Drive...
Successfully deleted conflicting /content/drive folder.
Mounted at /content/drive
Google Drive mounted successfully!
Environment setup completed successfully. OpenAI Client initialized and ready.


# Steps 3, 4, & 5: Page-by-Page Processing, Hybrid Analysis, and DOCX Assembly
Now, we define the core pipeline to process each PDF page, classify its content, analyze drawing pages via the GPT-4o multimodal API, and construct a beautifully formatted Word document with structured metadata tables.

In [42]:
def analyze_page_with_openai(image_bytes):
    """Calls OpenAI GPT-4o multimodal model to analyze and describe the engineering drawing."""
    if not client:
        return "[Simulation Mode: OpenAI API Key is missing. Unable to execute multimodal vision analysis for this engineering drawing page.]"

    # Encode image to base64
    base64_image = base64.b64encode(image_bytes).decode('utf-8')

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Analyze this engineering drawing and generate the Grounded Semantic Paragraph representing all elements and measurements detailed in the system guidelines."
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=2000,
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[Error during OpenAI API Call: {str(e)}]"

def clean_text(text):
    """Removes redundant spaces and normalizes line breaks."""
    if not text:
        return ""
    text = re.sub(r'[\r\n]+', '\n', text)
    text = re.sub(r' +', ' ', text)
    return text.strip()

def process_pdfs():
    pdf_files = glob.glob(os.path.join(INPUT_DIR, "*.pdf"))
    if not pdf_files:
        print(f"No PDF files found in {INPUT_DIR}. Please mount Google Drive or verify paths.")
        return

    print(f"Found {len(pdf_files)} PDF files to process.\n")

    for i, pdf_path in enumerate(pdf_files, 1):
        filename = os.path.basename(pdf_path)
        print(f"[{i}/{len(pdf_files)}] Starting file: {filename}")

        try:
            # Step 4: Initialize clean Word Document
            doc = Document()
            for section in doc.sections:
                section.top_margin = Inches(1)
                section.bottom_margin = Inches(1)
                section.left_margin = Inches(1)
                section.right_margin = Inches(1)

            title = doc.add_paragraph()
            title.add_run(f"RAG Corpus Document - Processed from: {filename}").bold = True
            title.paragraph_format.space_after = Inches(0.25)

            # Read PDF page count and characters
            reader = pypdf.PdfReader(pdf_path)
            total_pages = len(reader.pages)

            for idx in range(total_pages):
                page_num = idx + 1
                print(f"  - Processing Page {page_num}/{total_pages}")

                page = reader.pages[idx]
                raw_text = ""
                try:
                    raw_text = clean_text(page.extract_text())
                except Exception as text_err:
                    print(f"    * Direct text extraction failed or timed out: {text_err}")

                char_count = len(raw_text)

                # Step 2.B: Classification
                if char_count > 250:
                    classification = "Text-Heavy Page"
                elif char_count < 150:
                    classification = "Drawing-Only Page"
                else:
                    classification = "Hybrid Page (Text + Embedded Drawing)"

                # Generate unique chunk ID
                chunk_id = f"{os.path.splitext(filename)[0]}_P{page_num:03d}"

                # Step 4.B: Create Metadata Block (Table format)
                table = doc.add_table(rows=5, cols=2)
                table.style = 'Table Grid'

                metadata = [
                    ("Chunk_ID", chunk_id),
                    ("Source File", filename),
                    ("Page Number", str(page_num)),
                    ("Model/Processing", "GPT-4o (Vision) / Hybrid Pipeline" if classification != "Text-Heavy Page" else "Digital Extraction Pipeline"),
                    ("Engineering Classification", classification)
                ]

                for r_idx, (key, value) in enumerate(metadata):
                    table.cell(r_idx, 0).text = key
                    table.cell(r_idx, 1).text = value
                    # Bold the metadata keys
                    table.cell(r_idx, 0).paragraphs[0].runs[0].font.bold = True

                doc.add_paragraph().paragraph_format.space_after = Inches(0.1)

                # Step 3 & 4.C: Multimodal extraction & formatting based on classification
                if classification == "Text-Heavy Page":
                    doc.add_heading("Extracted Technical Text:", level=2)
                    doc.add_paragraph(raw_text)

                elif classification == "Drawing-Only Page":
                    doc.add_heading("Analyzed Engineering Drawing:", level=2)
                    try:
                        images = convert_from_path(pdf_path, first_page=page_num, last_page=page_num)
                        if images:
                            img_byte_arr = io.BytesIO()
                            images[0].save(img_byte_arr, format='JPEG')
                            analysis_result = analyze_page_with_openai(img_byte_arr.getvalue())
                            doc.add_paragraph(analysis_result)
                        else:
                            doc.add_paragraph("[Error: Could not render page image]")
                    except Exception as img_err:
                        doc.add_paragraph(f"[Error rendering drawing: {img_err}]")

                elif classification == "Hybrid Page (Text + Embedded Drawing)":
                    doc.add_heading("Extracted Explanatory Text:", level=2)
                    doc.add_paragraph(raw_text)

                    doc.add_heading(f"[Embedded Engineering Drawing Specification Analysis - Page {page_num}]", level=2)
                    try:
                        images = convert_from_path(pdf_path, first_page=page_num, last_page=page_num)
                        if images:
                            img_byte_arr = io.BytesIO()
                            images[0].save(img_byte_arr, format='JPEG')
                            analysis_result = analyze_page_with_openai(img_byte_arr.getvalue())
                            doc.add_paragraph(analysis_result)
                        else:
                            doc.add_paragraph("[Error: Could not render page image]")
                    except Exception as img_err:
                        doc.add_paragraph(f"[Error rendering drawing: {img_err}]")

                # Visual page delimiter
                doc.add_paragraph("__________________________________________________________________")
                doc.add_paragraph().paragraph_format.space_after = Inches(0.2)

            # Step 5: Save compiled structured DOCX
            output_filename = f"{os.path.splitext(filename)[0]}_structured.docx"
            output_filepath = os.path.join(OUTPUT_DIR, output_filename)
            doc.save(output_filepath)
            print(f"-> Successfully saved structured document to: {output_filepath}\n")

        except Exception as file_err:
            print(f"!!! Critical Error processing {filename}: {file_err}\n")

# Run the complete robust extraction
process_pdfs()

Found 1 PDF files to process.

[1/1] Starting file: plant_planning.pdf
  - Processing Page 1/192
  - Processing Page 2/192
  - Processing Page 3/192
  - Processing Page 4/192
  - Processing Page 5/192
  - Processing Page 6/192
  - Processing Page 7/192
  - Processing Page 8/192
  - Processing Page 9/192
  - Processing Page 10/192
  - Processing Page 11/192
  - Processing Page 12/192
  - Processing Page 13/192
  - Processing Page 14/192
  - Processing Page 15/192
  - Processing Page 16/192
  - Processing Page 17/192
  - Processing Page 18/192
  - Processing Page 19/192
  - Processing Page 20/192
  - Processing Page 21/192
  - Processing Page 22/192
  - Processing Page 23/192
  - Processing Page 24/192
  - Processing Page 25/192
  - Processing Page 26/192
  - Processing Page 27/192
  - Processing Page 28/192
  - Processing Page 29/192
  - Processing Page 30/192
  - Processing Page 31/192
  - Processing Page 32/192
  - Processing Page 33/192
  - Processing Page 34/192
  - Processing Page 